# Suite SKL — Agent guides

`AGENTS.md`, `llms.txt` and this package's skill, in `mbse/Patterns/skill/`, teach AI agents what this package is for
and how to use it. Each language has its own skill, written in that language, on the same topics in the same order.
Stale guidance misleads more than missing guidance, so these cases check that the two skills have the same headings in
the same order, that this skill's complete programs run, and that every link in the guides resolves.

In [ ]:
import re
import subprocess
import sys
import tempfile
from pathlib import Path

import mbse.Patterns as package

ROOT = Path.cwd().parent.parent
SKILL = Path(package.__file__).parent / "skill"
TYPESCRIPT = ROOT / "typescript5" / "skill"
REPO = "https://github.com/pitaman71/mbse-patterns/blob/main/"


def files(folder):
    """The files under `folder`, by relative path."""
    return {p.relative_to(folder).as_posix(): p.read_bytes() for p in sorted(folder.rglob("*")) if p.is_file()}


def headings(folder):
    """Each Markdown file's headings, outside fenced blocks, in order, by relative path."""
    return {name: re.findall(r"^#+ .*$", re.sub(r"^```.*?^```$", "", text.decode(), flags=re.M | re.S), re.M)
            for name, text in files(folder).items() if name.endswith(".md")}


GUIDES = [ROOT / "AGENTS.md", ROOT / "llms.txt", *(SKILL / name for name in files(SKILL) if name.endswith(".md"))]


def blocks(language):
    """The complete programs in the skill: fenced blocks whose info string is exactly `language`."""
    return [code for guide in GUIDES[2:] for code in re.findall(rf"^```{language}\n(.*?)^```$", guide.read_text(), re.M | re.S)]

## SKL-01 · The two languages' skills have the same headings, in the same order

In [ ]:
assert "SKILL.md" in files(SKILL) and "SKILL.md" in files(TYPESCRIPT)
assert headings(SKILL) == headings(TYPESCRIPT), (headings(SKILL), headings(TYPESCRIPT))

## SKL-02 · Every complete program in the skill runs

In [ ]:
programs = blocks("python")
assert programs, "the skill has no complete Python program"
failures = {}
with tempfile.TemporaryDirectory() as folder:
    for i, code in enumerate(programs):
        path = Path(folder) / f"skill_{i}.py"
        path.write_text(code)
        result = subprocess.run([sys.executable, str(path)], capture_output=True, text=True, timeout=300)
        if result.returncode != 0:
            failures[i] = result.stderr[-2000:]
assert failures == {}, failures

## SKL-03 · Every link in the agent guides resolves, anchors included

In [ ]:
def slug(heading):
    """GitHub's anchor for a heading."""
    return re.sub(r"[^\w\- ]", "", heading.strip().lower()).replace(" ", "-")

def anchors(text):
    """A document's anchors: its headings' and its items' `<a id>`s."""
    return {slug(h) for h in re.findall(r"^#+ (.*)$", text, re.M)} | set(re.findall(r'<a id="([^"]+)"></a>', text))


problems, checked = [], 0
for guide in GUIDES:
    for target in re.findall(r"\]\(([^)\s]+)\)", guide.read_text()):
        if sibling := re.fullmatch(r"(?:mbse:([^/]+)|https://github\.com/pitaman71/([^/]+)/blob/main)/(.*)", target):
            path, base = sibling[3], ROOT.parent / (sibling[1] or sibling[2])  # the sibling checkout, this one included
        elif "://" in target:
            continue
        else:
            path, base = target, guide.parent
        name, _, anchor = path.partition("#")
        resolved = base / name
        checked += 1
        if not resolved.exists():
            problems.append(f"{guide.name}: {target}")
        elif anchor and anchor not in anchors(resolved.read_text()):
            problems.append(f"{guide.name}: {target}")
assert problems == [] and checked > 0, (problems, checked)